In [5]:
"""
Mtb Q-Loop Pharmacophore Model — v2
=====================================
Improvements over v1 (selected from GPT review, with justification):

  P1  – Best-conformer selection: only the conformer with lowest alignment
        RMSD is used per molecule. Eliminates conformational overweighting.

  P4a – Per-family DBSCAN epsilon: tighter for polar features (Donor 1.2 Å,
        Acceptor/Aromatic 1.4 Å), wider for Hydrophobe (1.8 Å).

  P4b – min_samples enforced as unique molecules, not raw observation count.

  P5  – Soft coverage filter: keep if coverage ≥ 50 %, OR if coverage ≥ 30 %
        AND the cluster is tight (radius < 1.5 Å).

  P7  – Smarter intra-family dedup: tiebreak on spatial variance (prefer the
        tighter cluster) instead of coverage alone.

  P8  – Weighted linear scoring replaces Gaussian decay:
          contribution = weight × max(0, 1 − d / radius)
        Goes to exactly zero at the sphere boundary; per-family weights.

  P9  – Feature-atom sanity report: for each model point, lists which atom
        in each molecule satisfies it (so you can verify it's a real feature).

  P10 – Alignment sanity check: prints RMSD distribution and flags if
        cluster spread suggests a broken alignment.

Deliberately NOT implemented (with reasons):
  P2  – Redundant once P1 selects one conformer per molecule.
  P3  – Directional filtering requires docked poses or protein structure;
        RDKit feature positions are atom centres, not H-bond vector tips.
  P6  – Exclusion volumes only make sense for screening an external library;
        this pipeline scores training molecules, so they would be unused.
"""

import os
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS, ChemicalFeatures
from sklearn.cluster import DBSCAN
from collections import defaultdict

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CONFORMERS_DIR        = "Conformers"
TEMPLATE_NAME         = "CK_2_63.sdf"
EXCEL_PATH            = "Mtb.xlsx"

MIN_RADIUS            = 1.0   # Å — smallest pharmacophore sphere
MAX_RADIUS            = 2.5   # Å — largest pharmacophore sphere
ALIGNMENT_RMSD_CUTOFF = 5.0   # Å — discard alignments worse than this
INTRA_FAMILY_MIN_DIST = 2.0   # Å — merge same-family points closer than this

# P4a — per-family DBSCAN epsilon (Å)
FAMILY_EPS = {
    "Donor":      1.2,
    "Acceptor":   1.4,
    "Aromatic":   1.4,
    "Hydrophobe": 1.8,
    "Cationic":   1.2,
    "Anionic":    1.2,
}
DEFAULT_EPS = 1.4   # fallback for any family not listed above

# P5 — soft coverage thresholds
MIN_COVERAGE_HARD = 0.50   # always keep if coverage >= this
MIN_COVERAGE_SOFT = 0.30   # also keep if coverage >= this AND radius < SOFT_RADIUS_MAX
SOFT_RADIUS_MAX   = 1.5    # Å — tight-cluster threshold for the soft rule

# P8 — per-family scoring weights
FEATURE_WEIGHTS = {
    "Donor":      1.2,
    "Acceptor":   1.2,
    "Aromatic":   1.0,
    "Hydrophobe": 0.8,
    "Cationic":   1.3,
    "Anionic":    1.3,
}
DEFAULT_WEIGHT = 1.0

# ─────────────────────────────────────────────────────────────────────────────
# FEATURE DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
FDEF = """
DefineFeature Donor [$([N;!H0;v3,v4;+0,+1]),$(O[H]),$(S[H])]
  Family Donor
  Weights 1.0
EndFeature

DefineFeature Acceptor [$([N;H0;+0;v3]),$([O;H0;+0;v2]),$([F;$(F-[#6]);!$(FC[F,Cl,Br,I])]),$([S;H0;+0;v2])]
  Family Acceptor
  Weights 1.0
EndFeature

DefineFeature Aromatic [a]
  Family Aromatic
  Weights 1.0
EndFeature

DefineFeature Hydrophobe [$([c,s,br,I]),$([C;D3,D4;!$([C,N,O]=[C,N,O,S]);!$([CH2][O,N,S]);!$([CH][O,N,S]);!$([C][O,N,S])]),$([F,Cl,Br,I;$([F,Cl,Br,I]-[#6;!$([#6]~[#7,#8,#16])])])]
  Family Hydrophobe
  Weights 0.8
EndFeature

DefineFeature Cationic [$([NH2;+1]),$([NH3;+1]),$([NH4;+1]),$([nH;+1])]
  Family Cationic
  Weights 1.2
EndFeature

DefineFeature Anionic [$([C](=O)[O-]),$([P](=O)[O-]),$([S](=O)[O-]),$([c](=O)[O-])]
  Family Anionic
  Weights 1.2
EndFeature
"""

# ─────────────────────────────────────────────────────────────────────────────
# FILE / MOLECULE UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def find_sdf(name, directory):
    """Fuzzy-match a compound name to an SDF file (handles - / _ variants)."""
    target   = str(name).lower().strip().replace('.sdf', '')
    variants = {target, target.replace('-', '_'), target.replace('_', '-')}
    for filename in os.listdir(directory):
        if filename.lower().endswith('.sdf'):
            stem = filename.lower().replace('.sdf', '')
            if stem in variants or target in stem:
                return os.path.join(directory, filename)
    return None


def load_mol(path, name=None):
    """
    Load an SDF (may contain multiple conformers).
    Returns a single RDKit Mol with all conformers attached, or None.
    """
    suppl = Chem.SDMolSupplier(path, removeHs=False, sanitize=False)
    mols  = [m for m in suppl if m is not None]
    if not mols:
        return None

    base = mols[0]
    try:
        Chem.SanitizeMol(base)
    except Exception:
        try:
            Chem.SanitizeMol(
                base,
                Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE,
            )
        except Exception:
            return None

    for m in mols[1:]:
        try:
            base.AddConformer(m.GetConformer(), assignId=True)
        except Exception:
            continue

    if name:
        base.SetProp("_Name", name)
    return base


def align_to_template(mol, template):
    """
    Align mol to template via MCS.
    Returns (mol, per_conf_rmsd_dict) where keys are conformer IDs.
    Returns (mol, {}) if alignment fails.
    """
    mcs = rdFMCS.FindMCS([template, mol], timeout=2)
    if mcs.numAtoms < 3:
        return mol, {}

    patt = Chem.MolFromSmarts(mcs.smartsString)
    if patt is None:
        return mol, {}

    probe_match    = mol.GetSubstructMatch(patt)
    template_match = template.GetSubstructMatch(patt)
    if not probe_match or not template_match:
        return mol, {}

    atom_map   = list(zip(probe_match, template_match))
    conf_rmsds = {}

    for conf in mol.GetConformers():
        cid = conf.GetId()
        try:
            rmsd = rdMolAlign.AlignMol(
                mol, template,
                prbCid=cid, refCid=0,
                atomMap=atom_map,
            )
            conf_rmsds[cid] = rmsd
        except Exception:
            continue

    return mol, conf_rmsds


# ─────────────────────────────────────────────────────────────────────────────
# P1 — BEST-CONFORMER SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def select_best_conformer(mol, conf_rmsds):
    """
    Return a new Mol containing only the single conformer with the lowest
    alignment RMSD.  Each molecule therefore contributes exactly one vote
    to the global clustering step, regardless of how many conformers it had.
    """
    if not conf_rmsds:
        best_cid = mol.GetConformers()[0].GetId()
    else:
        best_cid = min(conf_rmsds, key=conf_rmsds.get)

    new_mol = Chem.RWMol(mol)
    for conf in mol.GetConformers():
        if conf.GetId() != best_cid:
            new_mol.RemoveConformer(conf.GetId())

    return new_mol.GetMol(), conf_rmsds.get(best_cid, 999.0)


# ─────────────────────────────────────────────────────────────────────────────
# PHARMACOPHORE MODEL BUILDING
# ─────────────────────────────────────────────────────────────────────────────

def _passes_coverage(coverage, radius):
    """P5 soft coverage rule. Returns (bool, rule_label)."""
    if coverage >= MIN_COVERAGE_HARD:
        return True, "hard"
    if coverage >= MIN_COVERAGE_SOFT and radius < SOFT_RADIUS_MAX:
        return True, "soft"
    return False, None


def build_pharmacophore(aligned_mols, factory):
    """
    Cluster feature positions and return model points that pass coverage.

    Each model point dict:
        family         – feature family name
        center         – 3-D centroid (np.array)
        radius         – sphere radius in Å
        coverage       – fraction of input molecules that contribute
        variance       – mean intra-cluster distance (compactness)
        coverage_rule  – 'hard' or 'soft'
        atom_idxs      – {mol_name: atom_index} for P9 sanity report
    """
    n_mols  = len(aligned_mols)
    all_obs = defaultdict(list)

    for mol_idx, mol in enumerate(aligned_mols):
        name  = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{mol_idx}"
        feats = factory.GetFeaturesForMol(mol)
        cid   = mol.GetConformers()[0].GetId()   # one conformer per mol after P1
        for f in feats:
            all_obs[f.GetFamily()].append({
                'pos':      np.array(f.GetPos(cid)),
                'mol_idx':  mol_idx,
                'mol_name': name,
                'atom_idx': f.GetAtomIds()[0] if f.GetAtomIds() else -1,
            })

    print(f"\n  Feature families detected: {sorted(all_obs.keys())}")

    candidates = []

    for family, obs in all_obs.items():
        if len(obs) < 2:
            continue

        coords = np.array([o['pos'] for o in obs])
        eps    = FAMILY_EPS.get(family, DEFAULT_EPS)

        # P4b — min_samples = min(2, unique molecules that have this feature)
        unique_in_family = len({o['mol_idx'] for o in obs})
        min_samp = min(2, unique_in_family)

        labels = DBSCAN(eps=eps, min_samples=min_samp).fit_predict(coords)

        for label in set(labels):
            if label == -1:
                continue

            indices      = np.where(labels == label)[0]
            u_mol_idxs   = {obs[i]['mol_idx'] for i in indices}

            # P4b — discard clusters that don't span at least 2 unique molecules
            if len(u_mol_idxs) < 2:
                continue

            cluster_coords = coords[indices]
            center   = cluster_coords.mean(axis=0)
            dists    = np.linalg.norm(cluster_coords - center, axis=1)
            variance = float(np.mean(dists))
            radius   = float(np.clip(variance * 1.5, MIN_RADIUS, MAX_RADIUS))
            coverage = len(u_mol_idxs) / n_mols
            atom_idxs = {obs[i]['mol_name']: obs[i]['atom_idx'] for i in indices}

            candidates.append({
                'family':    family,
                'center':    center,
                'radius':    radius,
                'coverage':  coverage,
                'variance':  variance,
                'atom_idxs': atom_idxs,
            })

    # ── P5 soft coverage filter ───────────────────────────────────────────────
    kept = []
    for p in candidates:
        ok, rule = _passes_coverage(p['coverage'], p['radius'])
        if ok:
            p['coverage_rule'] = rule
            kept.append(p)
        else:
            print(f"  Drop  {p['family']:<12}  coverage={p['coverage']:.0%}  "
                  f"radius={p['radius']:.2f}Å  (below soft threshold)")

    # ── P7 intra-family dedup: coverage desc, then variance asc ──────────────
    by_family = defaultdict(list)
    for p in kept:
        by_family[p['family']].append(p)

    final = []
    for family, pts in by_family.items():
        pts_sorted = sorted(pts, key=lambda x: (-x['coverage'], x['variance']))
        accepted   = []
        for pt in pts_sorted:
            too_close = any(
                np.linalg.norm(pt['center'] - ex['center']) < INTRA_FAMILY_MIN_DIST
                for ex in accepted
            )
            if not too_close:
                accepted.append(pt)
            else:
                print(f"  Merge {family:<12}  redundant point within "
                      f"{INTRA_FAMILY_MIN_DIST}Å")
        final.extend(accepted)

    return final


# ─────────────────────────────────────────────────────────────────────────────
# P8 — WEIGHTED LINEAR SCORING
# ─────────────────────────────────────────────────────────────────────────────

def score_molecule(mol, model, factory):
    """
    Weighted linear pharmacophore fit score in [0, 1].

        contribution_i = weight_i * max(0, 1 - d_i / radius_i)

    score = sum(contributions) / sum(weights)

    Returns (fit_score, n_hits).
    """
    if not model:
        return 0.0, 0

    total_weight = sum(FEATURE_WEIGHTS.get(p['family'], DEFAULT_WEIGHT) for p in model)
    weighted_sum = 0.0
    n_hits       = 0

    cid   = mol.GetConformers()[0].GetId()
    feats = factory.GetFeaturesForMol(mol)

    for pt in model:
        weight   = FEATURE_WEIGHTS.get(pt['family'], DEFAULT_WEIGHT)
        min_dist = np.inf

        for f in feats:
            if f.GetFamily() == pt['family']:
                d = np.linalg.norm(np.array(f.GetPos(cid)) - pt['center'])
                min_dist = min(min_dist, d)

        if min_dist < np.inf:
            weighted_sum += weight * max(0.0, 1.0 - min_dist / pt['radius'])
            if min_dist <= pt['radius']:
                n_hits += 1

    fit_score = weighted_sum / total_weight if total_weight > 0 else 0.0
    return float(fit_score), n_hits


# ─────────────────────────────────────────────────────────────────────────────
# P9 — FEATURE-ATOM SANITY REPORT
# ─────────────────────────────────────────────────────────────────────────────

def print_feature_atom_report(model, aligned_mols, factory):
    """
    For each model point, list which atom (index + element) in each molecule
    satisfies it (distance <= radius).  Warns if different atom types satisfy
    the same point across molecules, which may indicate the cluster is
    capturing conformational spread rather than a conserved chemical feature.
    """
    print(f"\n{'─'*70}")
    print("FEATURE–ATOM SANITY REPORT")
    print(f"{'─'*70}")

    for i, pt in enumerate(model, 1):
        cx, cy, cz = pt['center']
        print(f"\n  Point {i}: {pt['family']}  "
              f"({cx:.2f}, {cy:.2f}, {cz:.2f})  "
              f"r={pt['radius']:.2f}Å  cov={pt['coverage']:.0%} "
              f"[{pt.get('coverage_rule','?')}]")

        atom_symbols = []
        for mol in aligned_mols:
            name  = mol.GetProp("_Name") if mol.HasProp("_Name") else "?"
            cid   = mol.GetConformers()[0].GetId()
            feats = factory.GetFeaturesForMol(mol)

            best_d, best_sym, best_aidx = np.inf, None, None
            for f in feats:
                if f.GetFamily() == pt['family']:
                    d = np.linalg.norm(np.array(f.GetPos(cid)) - pt['center'])
                    if d < best_d:
                        best_d    = d
                        aidx      = f.GetAtomIds()[0] if f.GetAtomIds() else -1
                        best_aidx = aidx
                        best_sym  = (mol.GetAtomWithIdx(aidx).GetSymbol()
                                     if aidx >= 0 else "?")

            if best_d <= pt['radius']:
                print(f"    ✓ {name:<22}  atom {best_aidx:>3} ({best_sym})  "
                      f"d={best_d:.2f}Å")
                atom_symbols.append(best_sym)
            else:
                miss = f"d={best_d:.2f}Å" if best_d < np.inf else "no feature"
                print(f"    ✗ {name:<22}  MISS  ({miss})")

        unique_syms = set(atom_symbols)
        if len(unique_syms) > 1:
            print(f"    ⚠ Multiple atom types satisfying this point: {unique_syms} "
                  f"— verify chemical consistency")


# ─────────────────────────────────────────────────────────────────────────────
# P10 — ALIGNMENT SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────

def print_alignment_sanity(rmsd_values):
    """Print RMSD statistics and warn if spread is high."""
    if not rmsd_values:
        return
    arr = np.array(rmsd_values)
    print(f"\n{'─'*70}")
    print("ALIGNMENT SANITY CHECK")
    print(f"{'─'*70}")
    print(f"  n={len(arr)}  min={arr.min():.3f}  max={arr.max():.3f}  "
          f"mean={arr.mean():.3f}  std={arr.std():.3f} Å")

    if arr.max() > 3.0:
        print("  ⚠ Some alignments have RMSD > 3.0 Å — inspect those molecules")
    if arr.std() > 1.0:
        print("  ⚠ High RMSD spread (std > 1.0 Å) — cluster centres may be unreliable")
    if arr.max() <= 3.0 and arr.std() <= 1.0:
        print("  ✓ Alignment looks consistent")


# ─────────────────────────────────────────────────────────────────────────────
# XML EXPORT
# ─────────────────────────────────────────────────────────────────────────────

def save_model_xml(model, path="pharmacophore_model.xml"):
    """Save pharmacophore model as XML (LigandScout / Phase / PharmaGist compatible)."""
    import xml.etree.ElementTree as ET
    from xml.dom import minidom

    root = ET.Element("pharmacophore")
    root.set("name",       "Mtb_Q-Loop")
    root.set("created",    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))
    root.set("n_features", str(len(model)))

    for i, pt in enumerate(model, 1):
        cx, cy, cz = pt['center']
        feat = ET.SubElement(root, "feature")
        feat.set("id",     str(i))
        feat.set("family", pt['family'])

        cen = ET.SubElement(feat, "center")
        cen.set("x", f"{cx:.4f}")
        cen.set("y", f"{cy:.4f}")
        cen.set("z", f"{cz:.4f}")

        sph = ET.SubElement(feat, "sphere")
        sph.set("radius", f"{pt['radius']:.4f}")

        cov = ET.SubElement(feat, "coverage")
        cov.set("value", f"{pt['coverage']:.4f}")
        cov.set("rule",  pt.get('coverage_rule', '?'))

    raw    = ET.tostring(root, encoding="unicode")
    pretty = minidom.parseString(raw).toprettyxml(indent="  ")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(pretty)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def run():
    print("=" * 70)
    print("Mtb Q-Loop Pharmacophore Model v2")
    print("=" * 70)

    # ── Feature factory ───────────────────────────────────────────────────────
    factory = ChemicalFeatures.BuildFeatureFactoryFromString(FDEF)
    print("Feature factory built.")

    # ── Load data ─────────────────────────────────────────────────────────────
    df = pd.read_excel(EXCEL_PATH, sheet_name=1)
    df = df[df['Binding site'].str.upper() == 'Q-LOOP'].copy()

    ic50_col = pd.to_numeric(
        df['IC50 μM'].astype(str).str.replace('μM', '', regex=False),
        errors='coerce',
    )
    df['pIC50'] = -np.log10(ic50_col * 1e-6)
    df['IC50']  = ic50_col
    df = df.dropna(subset=['pIC50']).reset_index(drop=True)
    print(f"Compounds loaded: {len(df)}  "
          f"(IC50 {df['IC50'].min():.4f} – {df['IC50'].max():.4f} µM)")

    # ── Load template ─────────────────────────────────────────────────────────
    t_path = find_sdf(TEMPLATE_NAME, CONFORMERS_DIR)
    if not t_path:
        raise FileNotFoundError(f"Template '{TEMPLATE_NAME}' not found in '{CONFORMERS_DIR}'")
    template = load_mol(t_path)
    if template is None:
        raise RuntimeError("Could not load template molecule")
    print(f"Template: {TEMPLATE_NAME}")

    # ── Align + P1 best-conformer selection ───────────────────────────────────
    print("\nAligning compounds (selecting best conformer per molecule)...")
    aligned_mols = []
    failed       = []
    rmsd_values  = []

    for _, row in df.iterrows():
        path = find_sdf(row['Name'], CONFORMERS_DIR)
        if not path:
            print(f"  {row['Name']:<22}  NOT FOUND")
            failed.append(row['Name'])
            continue

        mol = load_mol(path, row['Name'])
        if mol is None:
            print(f"  {row['Name']:<22}  LOAD ERROR")
            failed.append(row['Name'])
            continue

        n_confs         = mol.GetNumConformers()
        mol, conf_rmsds = align_to_template(mol, template)

        if not conf_rmsds:
            print(f"  {row['Name']:<22}  ALIGNMENT FAILED  (MCS < 3 atoms)")
            failed.append(row['Name'])
            continue

        mol, best_rmsd = select_best_conformer(mol, conf_rmsds)

        if best_rmsd > ALIGNMENT_RMSD_CUTOFF:
            print(f"  {row['Name']:<22}  poor alignment  RMSD={best_rmsd:.2f}Å  (skipped)")
            failed.append(row['Name'])
            continue

        print(f"  {row['Name']:<22}  RMSD={best_rmsd:.3f}Å  "
              f"(best of {n_confs} conformer{'s' if n_confs > 1 else ''})")
        aligned_mols.append(mol)
        rmsd_values.append(best_rmsd)

    print(f"\nAligned: {len(aligned_mols)}/{len(df)} compounds  "
          f"({len(failed)} skipped)")

    if len(aligned_mols) < 3:
        raise RuntimeError("Need at least 3 aligned molecules to build a model")

    # P10 — alignment sanity
    print_alignment_sanity(rmsd_values)

    # ── Build model ───────────────────────────────────────────────────────────
    print("\nBuilding pharmacophore model...")
    model = build_pharmacophore(aligned_mols, factory)

    if not model:
        raise RuntimeError("No pharmacophore features survived — "
                           "try lowering MIN_COVERAGE_SOFT or SOFT_RADIUS_MAX")

    # ── Print model table ─────────────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"PHARMACOPHORE MODEL  ({len(model)} features)")
    print(f"{'─'*70}")
    print(f"{'#':<4} {'Family':<12} {'X':>8} {'Y':>8} {'Z':>8} "
          f"{'Radius':>8} {'Coverage':>10} {'Rule':>6}")
    print(f"{'─'*70}")
    for i, pt in enumerate(model, 1):
        cx, cy, cz = pt['center']
        print(f"{i:<4} {pt['family']:<12} {cx:>8.2f} {cy:>8.2f} {cz:>8.2f} "
              f"{pt['radius']:>8.2f} {pt['coverage']:>9.1%} "
              f"{pt.get('coverage_rule','?'):>6}")
    print(f"{'─'*70}")
    print(f"Feature families: {sorted({p['family'] for p in model})}")

    # P9 — feature-atom sanity report
    print_feature_atom_report(model, aligned_mols, factory)

    # ── Fit scores ────────────────────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print("FIT SCORES")
    print(f"{'─'*70}")
    print(f"{'Name':<22} {'IC50 (µM)':>11} {'Fit Score':>10} {'Hits':>8}")
    print(f"{'─'*70}")

    results = []
    for mol in aligned_mols:
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else "unknown"
        ic50 = df.loc[df['Name'] == name, 'IC50']
        ic50 = float(ic50.iloc[0]) if len(ic50) else float('nan')

        fit, hits = score_molecule(mol, model, factory)
        results.append({'name': name, 'ic50': ic50, 'fit': fit, 'hits': hits})

    for r in sorted(results, key=lambda x: -x['fit']):
        hit_str  = f"{r['hits']}/{len(model)}"
        ic50_str = f"{r['ic50']:.4f}" if not np.isnan(r['ic50']) else "N/A"
        print(f"{r['name']:<22} {ic50_str:>11} {r['fit']:>10.3f} {hit_str:>8}")

    print(f"{'─'*70}")

    # ── Save XML ──────────────────────────────────────────────────────────────
    xml_path = "pharmacophore_model.xml"
    save_model_xml(model, xml_path)
    print(f"\nPharmacophore model saved → {xml_path}")
    print("\nDone.")


if __name__ == "__main__":
    run()

Mtb Q-Loop Pharmacophore Model v2
Feature factory built.
Compounds loaded: 22  (IC50 0.0030 – 15.8000 µM)
Template: CK_2_63.sdf

Aligning compounds (selecting best conformer per molecule)...
  Aurachin D              RMSD=0.846Å  (best of 11 conformers)
  CK-3-22 (1T)            RMSD=0.634Å  (best of 32 conformers)
  CK-3-14                 RMSD=0.027Å  (best of 16 conformers)
  RKA-259                 NOT FOUND
  RKA-307                 RMSD=0.004Å  (best of 8 conformers)
  RKA-310                 RMSD=0.005Å  (best of 10 conformers)
  MTD-403                 RMSD=0.008Å  (best of 8 conformers)
  CK-2-88                 RMSD=0.003Å  (best of 4 conformers)
  CK-3-23                 RMSD=0.038Å  (best of 8 conformers)
  CK-2-63                 RMSD=0.000Å  (best of 16 conformers)
  PG-203                  RMSD=1.028Å  (best of 16 conformers)
  RKA-70                  RMSD=0.169Å  (best of 8 conformers)
  RKA-73                  RMSD=0.114Å  (best of 8 conformers)
  LT-9                 